## Middleware

Middleware provides a way to tightly control what happens inside an Agent
<br>
> Tracking agent behaviour with logging ,analytics and debugging

> Transforming prompts ,tool selection and output formatting

> Adding retries,fallbacks and early termination logic

> Applying rate limits,guardrails, and PII detection

In [1]:
import os 
from dotenv import load_dotenv
load_dotenv()

True

In [8]:
os.environ['GOOGLE_API_KEY']=os.getenv('GOOGLE_API_KEY')
from langchain_google_genai import ChatGoogleGenerativeAI

model=ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")

model

ChatGoogleGenerativeAI(profile={'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), model='gemini-2.5-flash-lite', client=<google.genai.client.Client object at 0x7f4cd4654370>, default_metadata=(), model_kwargs={})

In [9]:
model.invoke("Hello")

AIMessage(content='Hello! How can I help you today?', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d34ef-896b-7611-8a2b-a9ffb950d226-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 2, 'output_tokens': 9, 'total_tokens': 11, 'input_token_details': {'cache_read': 0}})

In [10]:
## Summarization Middleware
""" 
Automatically summarizing the converation history before reaching the context size and token limits.
To preserve recent messages while compressing older context.
"""
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import AIMessage,HumanMessage,SystemMessage

## message based summarization
agent=create_agent(
    model=model,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("messages",10),
            keep=("messages",4)
            
        )
    ]
)



In [20]:
##for langgraph --need to use Configurable
config={"configurable":{"thread_id":"session123"}}
config

{'configurable': {'thread_id': 'session123'}}

In [23]:
## messages
questions=[
    "what is 1+1?",
    "In which direction sun rises?",
    "What is the Capital of India?",
    "Which is national bird of America?",
    "Which is the Global Language?",
    "What is your model name?"

]

for q in questions:
    response=agent.invoke({"messages":[HumanMessage(content=q)]},
                          config=config)
    print(f"Messages:{response}")
    print(f"Messages:{len(response['messages'])}")


Messages:{'messages': [HumanMessage(content='Here is a summary of the conversation to date:\n\n## SESSION INTENT\nTo answer user\'s factual questions.\n\n## SUMMARY\nThe user has asked several factual questions, and the AI has provided correct answers. The last question asked was "Which is national bird of America?".\n\n## ARTIFACTS\nNone\n\n## NEXT STEPS\nAnswer the question "Which is the national bird of America?".', additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='18323f73-be82-417b-87cd-045324952590'), AIMessage(content='The national bird of the United States of America is the **Bald Eagle**.', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d3518-5d33-7f93-bc29-5ec7ca746334-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 58, 'output_tokens': 15, 'total_tokens': 73, 'input_token_details': {'cache_read':

In [24]:
## we can see the message are summarised after 10

In [ ]:
# Lets limit the messages based on the tokens
from langchain.tools import tool

#lets use groq model
from langchain_groq import ChatGroq
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model=ChatGroq(model="llama-3.3-70b-versatile")

@tool
def search_hotels(city:str) -> str:
    """ search hotels -return more details to exceed the limit set"""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""

agent=create_agent(
    model=model,
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("tokens",550),
            keep=("tokens",200)
        )
    ]
)

config={"configurable":{"thread_id":"test-2"}} ##this is fixed name


In [43]:

#token counter
cities=["Paris","London","Tokyo","India","Singapore","Delhi","Goa"]
##For agent pass the messages and the config 

def count_tokens(messages):
    total_chars=sum(len(str((m.content))) for m in messages)
    return total_chars//4  #4 chars - 1 token

for city in cities:
    response=agent.invoke(
        {
            "messages":[HumanMessage(content=f"Find Hotels in {city}")]
        },
        config=config
    )
    tokens=count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, Number of messages: {len(response['messages'])}")
    print(f"{response['messages']}")


Paris: ~293 tokens, Number of messages: 7
[HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\nThe user's primary goal is to find hotels, with the current city of interest being London.\n\n## SUMMARY\nThe user initially searched for hotels in Paris, considering various options, before changing the city to London. In London, the user was presented with hotel options including Grand Hotel, City Inn, and Budget Stay.\n\n## ARTIFACTS\nNone\n\n## NEXT STEPS\nThe next step is to refine the hotel search in London based on the user's preferences, such as price range or specific amenities, to provide a more tailored set of hotel options for the user to consider.", additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='cfb8d1aa-18d8-458c-9f56-f4daa2272c5f'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'd10dnf510', 'function': {'arguments': '{"city":"London"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kkbnd2zmerfr2f2znqakqn0s` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99496, Requested 697. Please try again in 2m46.752s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [48]:

#Fraction of  counter
# Lets limit the messages based on the tokens
from langchain.tools import tool

#lets use groq model
from langchain_groq import ChatGroq
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model=ChatGroq(model="qwen/qwen3-32b")

@tool
def search_hotels(city:str) -> str:
    """ search hotels -return more details to exceed the limit set"""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""

agent=create_agent(
    model=model,
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("fraction",0.005),
            keep=("fraction",0.002)
        )
    ]
)

config={"configurable":{"thread_id":"test-5"}} ##this is fixed name

cities=["Paris","London","Tokyo","India","Singapore","Delhi","Goa"]
##For agent pass the messages and the config 

def count_tokens(messages):
    total_chars=sum(len(str((m.content))) for m in messages)
    return total_chars//4  #4 chars - 1 token

for city in cities:
    response=agent.invoke(
        {
            "messages":[HumanMessage(content=f"Find Hotels in {city}")]
        },
        config=config
    )
    tokens=count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, Number of messages: {len(response['messages'])}")
    print(f"{response['messages']}")


Paris: ~148 tokens, Number of messages: 4
[HumanMessage(content='Find Hotels in Paris', additional_kwargs={}, response_metadata={}, id='11172c5c-7572-4320-a562-b7ca4a3a922e'), AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user wants to find hotels in Paris. Let me check the available tools. There\'s a function called search_hotels that takes a city parameter. Since the user specified Paris, I need to call this function with "Paris" as the city argument. I\'ll make sure the JSON is correctly formatted with the city parameter. No other parameters are needed since the function only requires the city. Let me double-check the syntax to avoid any errors.\n', 'tool_calls': [{'id': 'r9yaex5fx', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 116, 'prompt_tokens': 156, 'total_tokens': 272, 'completion_time': 0.180616398, 'completion_tokens_details': {'reasoning_token

In [ ]:
# the model is hallucinating and giving same city name as it is summarizing the context and 
# getting same info for all cities, so it is taking it into consideration



## Human in the Loop Middleware
Pause agent execution for human approval,editing and rejection of tool call befor they execute
- High stakes transaction
- Compliance workflows where human oversight is mandatory
- Lomg running conversations where human feedback guides the agent

In [51]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware

def read_email_tool(email_id:str)->str:
    """Mock function to read an email by its ID """
    return f"Email content for ID :{email_id}"
def send_email_tool(recipient:str,subject:str, body:str) ->str:
    """ Mock function to send an email"""
    return f"Email sent to {recipient} with subject :{subject}"

In [52]:
# creating the Human in middle

agent=create_agent(
    model=model,
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False
            }
        )
    ]
)

In [53]:
config={"configurable":{"thread_id":"test123"}}
#step1:request

result=agent.invoke(
    {"messages":[HumanMessage(content="Send email to ravi123@gmail.com with subject hello and body how are you?")]},
    config=config
)

result

{'messages': [HumanMessage(content='Send email to ravi123@gmail.com with subject hello and body how are you?', additional_kwargs={}, response_metadata={}, id='207ad795-34fe-442f-944b-c9e78b14fd29'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user wants to send an email to ravi123@gmail.com with the subject "hello" and the body "how are you?". Let me check the available tools. There\'s the send_email_tool which requires recipient, subject, and body. All three are required. The parameters are all strings. The user provided all three: recipient is the email address given, subject is "hello", and body is "how are you?". So I need to call send_email_tool with these arguments. No need to use read_email_tool here since the task is about sending, not reading. Just structure the JSON with the parameters as specified.\n', 'tool_calls': [{'id': 'sn0f22e85', 'function': {'arguments': '{"body":"how are you?","recipient":"ravi123@gmail.com","subject":"hello"}', 'name'

In [55]:
from langgraph.types import  Command
## step2 approve

if "__interrupt__" in result:
    print("&&&& Paused ! Approving...")

    result=agent.invoke(
        Command(
            resume={
                "decisions":[
                    {"type":"approve"}
                ]
            }
        ),
        config=config


    )
    print(f"!!!!Result:{result['messages'][-1].content}")

&&&& Paused ! Approving...
!!!!Result:The email has been successfully sent to ravi123@gmail.com with the subject "hello". Let me know if you need anything else!


In [56]:
result

{'messages': [HumanMessage(content='Send email to ravi123@gmail.com with subject hello and body how are you?', additional_kwargs={}, response_metadata={}, id='207ad795-34fe-442f-944b-c9e78b14fd29'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user wants to send an email to ravi123@gmail.com with the subject "hello" and the body "how are you?". Let me check the available tools. There\'s the send_email_tool which requires recipient, subject, and body. All three are required. The parameters are all strings. The user provided all three: recipient is the email address given, subject is "hello", and body is "how are you?". So I need to call send_email_tool with these arguments. No need to use read_email_tool here since the task is about sending, not reading. Just structure the JSON with the parameters as specified.\n', 'tool_calls': [{'id': 'sn0f22e85', 'function': {'arguments': '{"body":"how are you?","recipient":"ravi123@gmail.com","subject":"hello"}', 'name'

In [57]:
## Lets try the reject mail
config={"configurable":{"thread_id":"test-reject"}}
#step1:request

result=agent.invoke(
    {"messages":[HumanMessage(content="Send email to ravi123@gmail.com with subject hello and body how are you?")]},
    config=config
)



In [58]:
from langgraph.types import  Command
## step2 reject

if "__interrupt__" in result:
    print("&&&& Paused ! Approving...")

    result=agent.invoke(
        Command(
            resume={
                "decisions":[
                    {"type":"reject"}
                ]
            }
        ),
        config=config


    )
    print(f"!!!!Result:{result['messages'][-1].content}")

&&&& Paused ! Approving...
!!!!Result:The email could not be sent at this time. Please check the details or try again later. If the issue persists, consider reaching out for technical support.


In [59]:
result

{'messages': [HumanMessage(content='Send email to ravi123@gmail.com with subject hello and body how are you?', additional_kwargs={}, response_metadata={}, id='4dafce04-7fcf-4e9e-b80d-8f35e227ed08'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user wants to send an email to ravi123@gmail.com with the subject "hello" and the body "how are you?". Let me check the available tools. There\'s a send_email_tool that requires recipient, subject, and body. The parameters are all required, so I need to make sure all three are included. The function name is send_email_tool, so I\'ll structure the tool_call with those parameters. Let me verify the JSON format to avoid errors. Yep, that should do it.\n', 'tool_calls': [{'id': 'kr4m38bka', 'function': {'arguments': '{"body":"how are you?","recipient":"ravi123@gmail.com","subject":"hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 153, 'prompt_tokens': 25

In [61]:
## lets try the editing
## Lets try the reject mail
config={"configurable":{"thread_id":"test-edit"}}
#step1:request

result=agent.invoke(
    {"messages":[HumanMessage(content="Send email to ravi123@gmail.com with subject hello and body how are you?")]},
    config=config
)
result


{'messages': [HumanMessage(content='Send email to ravi123@gmail.com with subject hello and body how are you?', additional_kwargs={}, response_metadata={}, id='8bba2903-6a4b-4140-b3da-87b71504dc17'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user wants to send an email to ravi123@gmail.com with the subject "hello" and the body "how are you?". Let me check the available tools.\n\nLooking at the tools provided, there\'s a send_email_tool. Its parameters are recipient, subject, body. All three are required. The user provided all three: recipient is the email address given, subject is "hello", and body is "how are you?".\n\nSo I need to call send_email_tool with those arguments. No need to use read_email_tool here since the task is about sending, not reading. The parameters match exactly, so the function call should be straightforward. Just format the JSON correctly within the tool_call tags.\n', 'tool_calls': [{'id': 'ar8t1vy86', 'function': {'arguments': '

In [62]:
from langgraph.types import  Command
## step2 edit and approve

if "__interrupt__" in result:
    print("&&&& Paused ! Editing...")

    result=agent.invoke(
        Command(
            resume={
                "decisions":[
                    {"type":"edit",
                     "edited_action":{
                         "name":"send_email_tool",
                         "args":{
                             "recipient":"correct@email.com",
                             "subject":"Corrected subject",
                             "body":"This was edited by human"
                         }
                     }
                     
                     
                     
                     }
                ]
            }
        ),
        config=config


    )
    print(f"!!!!Result:{result['messages'][-1].content}")

&&&& Paused ! Editing...
!!!!Result:The email has been successfully sent to **correct@email.com** with the subject **"Corrected subject"** and the body **"This was edited by human"**. Let me know if you need to send another email or make further adjustments!


In [63]:
result

{'messages': [HumanMessage(content='Send email to ravi123@gmail.com with subject hello and body how are you?', additional_kwargs={}, response_metadata={}, id='8bba2903-6a4b-4140-b3da-87b71504dc17'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user wants to send an email to ravi123@gmail.com with the subject "hello" and the body "how are you?". Let me check the available tools.\n\nLooking at the tools provided, there\'s a send_email_tool. Its parameters are recipient, subject, body. All three are required. The user provided all three: recipient is the email address given, subject is "hello", and body is "how are you?".\n\nSo I need to call send_email_tool with those arguments. No need to use read_email_tool here since the task is about sending, not reading. The parameters match exactly, so the function call should be straightforward. Just format the JSON correctly within the tool_call tags.\n', 'tool_calls': [{'id': 'ar8t1vy86', 'function': {'arguments': '